# 少样本元学习：Omniglot、Mini-ImageNet 与 MAML

如果每个新类别只有几张带标签的图像，怎样让模型尽快学会区分它们？模型无关元学习（MAML）在许多小任务上训练，寻找一组经过少量梯度更新就能适应新任务的初始参数。FO-MAML 用一阶近似简化这一训练过程。

本节先实现任务采样、任务内适应和跨任务更新，再比较不同的任务大小。默认使用 Omniglot；将 `DATASET_NAME` 改为 `"mini-imagenet"` 后，可用相同流程训练彩色图像。两套数据的训练、验证与测试类别均互不重叠。

[本课说明与运行步骤](README.md)


In [ ]:
from biai.paths import DATA_DIR
from biai.reproducibility import seed_everything, parameter_digest

SEED = 0
seed_everything(SEED, deterministic=True)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict, defaultdict
import random
import time
import os
import copy
from PIL import Image
import torchvision.transforms as transforms
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
from pathlib import Path
import pickle
import codecs
from torchvision.datasets.utils import download_url, check_integrity

DATASET_NAME = "omniglot"


## 采样少样本分类任务

一个 N-way K-shot 任务包含 N 个类别，每类取 K 张带标签的图像作为**支持集**，用于调整当前任务的参数。每类另外取 `q_query` 张图像作为**查询集**，检查调整后的分类效果。支持集与查询集没有重复图像，标签在任务内重新编号为 0 到 N−1。这样的一次任务采样也称为一个 episode。

Omniglot 中，每个类别是一种字符。数据通过 [torchvision Omniglot](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.Omniglot.html) 下载到 `data/omniglot-py/`。代码将 `background` 中约 80% 的字符类分给训练集，其余分给验证集；`evaluation` 的字符类留作最终测试。图像先缩放到 28 × 28，再转换为 [0, 1] 的像素值并反转黑白，使笔画取高值、白色背景取零。采样器索引文件路径，在抽到任务时读取图像。

Mini-ImageNet 使用 [learn2learn 发布的数据](https://zenodo.org/records/7978538)：100 个类别分为 64 个训练类、16 个验证类和 20 个测试类，每类 600 张 84 × 84 的 RGB 图像。按类别划分，才能检验对新类别的适应能力；如果只是随机划分同一类别的图片，测试的问题就变了。图像像素在读入后缩放到 −1 到 1。

Mini-ImageNet 的三个缓存文件共约 1.8 GB，首次使用会自动下载并校验。也可以提前将 `mini-imagenet-cache-train.pkl`、`mini-imagenet-cache-validation.pkl` 和 `mini-imagenet-cache-test.pkl` 放到 `data/mini-imagenet/`。


### 从类别中抽取一个任务

先看任务采样：每类抽取互不重复的支持图像和查询图像，再把类别重新编号。返回的两组图像形状分别为 `[N × K, C, H, W]` 和 `[N × Q, C, H, W]`；标签分别有 `N × K` 和 `N × Q` 个。这里 `Q` 是每类查询图像数，`C、H、W` 是图像的通道数、高和宽。


In [ ]:
class FewShotEpisodes:
    def sample_episode(self, n_way, k_shot, q_query, rng=None):
        """Sample an episode with disjoint support and query sets.

        Choose n_way classes and relabel them from 0 to n_way - 1.
        Each class contributes k_shot support images and q_query query images.
        Return support images, support labels, query images, and query labels.
        """
        # Episode labels follow the order of the sampled classes.
        rng = self.rng if rng is None else rng
        chosen_classes = rng.sample(self.classes, n_way)

        support_images, support_labels = [], []
        query_images, query_labels = [], []

        for i, class_key in enumerate(chosen_classes):
            all_images = self.class_to_images[class_key]

            if len(all_images) < k_shot + q_query:
                raise ValueError(
                    f"类别 {class_key} 只有 {len(all_images)} 张图，无法提供 {k_shot + q_query} 个不同样本"
                )

            # Sample without replacement so support and query images are disjoint.
            selected_images = rng.sample(all_images, k_shot + q_query)
            support_paths = selected_images[:k_shot]
            query_paths = selected_images[k_shot:]

            for img_path in support_paths:
                img = self._load_image(img_path)
                support_images.append(img.unsqueeze(0))
                support_labels.append(i)

            for img_path in query_paths:
                img = self._load_image(img_path)
                query_images.append(img.unsqueeze(0))
                query_labels.append(i)

        return (
            torch.cat(support_images, dim=0),
            torch.LongTensor(support_labels),
            torch.cat(query_images, dim=0),
            torch.LongTensor(query_labels),
        )


### 按字符类别组织 Omniglot

下面建立“字符类别到图像路径”的索引。采样器抽到路径后，才读取对应图像。`subset_classes()` 用于把训练字符类与验证字符类分开。


In [ ]:
class OmniglotFewShot(FewShotEpisodes):
    def __init__(self, root=DATA_DIR / "omniglot-py", split="background", img_size=28):
        """Index one Omniglot split and prepare square grayscale images."""
        self.root = root
        self.split = split
        self.img_size = img_size
        self.input_channels = 1

        if split == "background":
            self.data_dir = os.path.join(root, "images_background")
        elif split == "evaluation":
            self.data_dir = os.path.join(root, "images_evaluation")
        else:
            raise ValueError(f"split 必须是 'background' 或 'evaluation', 得到: {split}")

        self.transform = transforms.Compose(
            [
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                # Make dark character strokes positive on a zero-valued background.
                transforms.Lambda(lambda image: 1.0 - image),
            ]
        )

        # Index image paths; images are opened when an episode is sampled.
        self.class_to_images = defaultdict(list)
        self._load_data()
        self.classes = list(self.class_to_images.keys())
        self.rng = random.Random(SEED)

        print(
            f"加载 {split} 数据集: {len(self.classes)} 个字符类, "
            f"共 {sum(len(imgs) for imgs in self.class_to_images.values())} 张图片"
        )

    def _load_data(self):
        """Group image paths by alphabet and character directory."""
        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(
                f"数据目录不存在: {self.data_dir}，请先执行本节的数据下载单元格。"
            )

        for alphabet in sorted(os.listdir(self.data_dir)):
            alphabet_path = os.path.join(self.data_dir, alphabet)
            if not os.path.isdir(alphabet_path) or alphabet.startswith("."):
                continue

            for character_dir in sorted(os.listdir(alphabet_path)):
                character_path = os.path.join(alphabet_path, character_dir)
                if not os.path.isdir(character_path) or not character_dir.startswith("character"):
                    continue

                # Character directory names can repeat across alphabets.
                class_key = (alphabet, character_dir)

                for img_file in sorted(os.listdir(character_path)):
                    if img_file.endswith(".png"):
                        img_path = os.path.join(character_path, img_file)
                        self.class_to_images[class_key].append(img_path)

    def _load_image(self, img_path):
        """Load one grayscale image and apply the configured transform."""
        img = Image.open(img_path).convert("L")
        img = self.transform(img)
        return img

    def subset_classes(self, classes):
        """Share image paths while keeping the selected character classes separate."""
        subset = copy.copy(self)
        subset.classes = list(classes)
        subset.class_to_images = {key: self.class_to_images[key] for key in classes}
        subset.rng = random.Random(SEED)
        return subset


### 读取 Mini-ImageNet 缓存

Mini-ImageNet 的图像存放在 NumPy 数组中，类别索引指向数组位置。下面读取并校验发布的数据缓存，再复用同一个任务采样器。使用默认 Omniglot 时，这部分类定义不会下载或加载 Mini-ImageNet。


In [ ]:
class MiniImageNetUnpickler(pickle.Unpickler):
    """Read the published NumPy cache without permitting arbitrary Python classes."""

    def find_class(self, module, name):
        allowed = {
            ("numpy.core.multiarray", "_reconstruct"): np._core.multiarray._reconstruct,
            ("numpy._core.multiarray", "_reconstruct"): np._core.multiarray._reconstruct,
            ("numpy", "ndarray"): np.ndarray,
            ("numpy", "dtype"): np.dtype,
            ("_codecs", "encode"): codecs.encode,
        }
        if (module, name) not in allowed:
            raise pickle.UnpicklingError(f"Unsupported cache type: {module}.{name}")
        return allowed[(module, name)]


class MiniImageNetFewShot(FewShotEpisodes):
    # Checksums published with learn2learn's fixed Zenodo dataset record.
    CHECKSUMS = {
        "train": "ee4e80dc7b716f3ceec891d199b5277a",
        "validation": "c1145a80890b88d54a67d6e0022d53a5",
        "test": "389b6c59d348b2e1ae9610f00a5e6724",
    }

    def __init__(self, root=DATA_DIR / "mini-imagenet", split="train"):
        self.root, self.split = Path(root), split
        self.input_channels, self.img_size = 3, 84
        filename = f"mini-imagenet-cache-{split}.pkl"
        checksum = self.CHECKSUMS[split]
        path = self.root / filename
        if not check_integrity(path, checksum):
            download_url(
                f"https://zenodo.org/records/7978538/files/{filename}",
                self.root,
                filename=filename,
                md5=checksum,
            )
        with path.open("rb") as stream:
            cache = MiniImageNetUnpickler(stream, encoding="latin1").load()
        self.images = cache["image_data"]
        self.class_to_images = cache["class_dict"]
        self.classes = sorted(self.class_to_images)
        indices = [i for values in self.class_to_images.values() for i in values]
        if (
            self.images.dtype != np.uint8
            or self.images.shape[1:] != (84, 84, 3)
            or sorted(indices) != list(range(len(self.images)))
        ):
            raise ValueError("Mini-ImageNet cache must partition RGB 84x84 images by class")
        self.rng = random.Random(SEED)
        self.transform = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)]
        )
        print(f"Mini-ImageNet {split}: {len(self.classes)} classes, {len(self.images)} images")

    def _load_image(self, index):
        return self.transform(self.images[index])


### 建立训练、验证和测试集

`load_few_shot_data()` 返回三份按类别分开的数据。训练集用于更新参数，验证集用于选择参数，测试集用于评估对新类别的适应。


In [ ]:
def load_few_shot_data(name):
    if name == "mini-imagenet":
        splits = tuple(
            MiniImageNetFewShot(split=split) for split in ("train", "validation", "test")
        )
        if tuple(len(ds.classes) for ds in splits) != (64, 16, 20):
            raise ValueError("Mini-ImageNet requires the 64/16/20 class split")
    elif name == "omniglot":
        for background in (True, False):
            datasets.Omniglot(root=DATA_DIR, background=background, download=True)
        background_ds = OmniglotFewShot(split="background")
        class_order = list(background_ds.classes)
        random.Random(SEED).shuffle(class_order)
        n_validation_classes = max(2, len(class_order) // 5)
        splits = (
            background_ds.subset_classes(class_order[n_validation_classes:]),
            background_ds.subset_classes(class_order[:n_validation_classes]),
            OmniglotFewShot(split="evaluation"),
        )
    else:
        raise ValueError(f"Unknown dataset: {name}")
    # Omniglot keys contain alphabet names; Mini-ImageNet keys are WordNet IDs.
    class_sets = [set(ds.classes) for ds in splits]
    if any(class_sets[i] & class_sets[j] for i in range(3) for j in range(i + 1, 3)):
        raise ValueError("Training, validation and test classes must be disjoint")
    return splits


In [ ]:
train_ds, val_ds, test_ds = load_few_shot_data(DATASET_NAME)
print(
    f"Classes: train={len(train_ds.classes)}, "
    f"validation={len(val_ds.classes)}, test={len(test_ds.classes)}"
)


## 共享的卷积网络

网络由四个卷积块组成，每个块包含卷积、组归一化（GroupNorm）、ReLU 和最大池化。Omniglot 灰度图使用 1 个输入通道，Mini-ImageNet 彩色图使用 3 个。经过四次池化和一次全局平均池化后，两种输入都得到 64 维特征，再由线性层输出当前任务的 N 个类别分数。

组归一化将每层的 64 个通道分成 8 组，在每张图像内部计算均值和方差。在默认的 5-way 1-shot 任务中，支持集只有 5 张图像；这种归一化方式不依赖批次大小，也不保存跨任务的运行统计量。因此，同一张查询图像的预测不会随同批其他图像改变。归一化的缩放与偏移参数和卷积参数一起学习。

`forward()` 使用模型自身的参数；`functional_forward()` 接收外部传入的一组参数。后者让每个任务使用自己的适应结果，而无需覆盖共享的初始参数。


In [ ]:
class MAML_CNN(nn.Module):
    """Four convolution-pooling blocks supporting grayscale and RGB images."""

    def __init__(self, n_way, input_channels=1, img_size=28):
        super().__init__()
        self.n_way = n_way

        # Global pooling gives 64 features for both 28x28 and 84x84 inputs.
        self.final_feature_dim = 64

        # GroupNorm normalizes each image independently, without running statistics.
        # Keep layer names aligned with the keys used by functional_forward.
        self.layers = OrderedDict(
            {
                "conv1": nn.Conv2d(input_channels, 64, kernel_size=3, padding=1),
                "norm1": nn.GroupNorm(8, 64),
                "relu1": nn.ReLU(),
                "pool1": nn.MaxPool2d(kernel_size=2),  # 28x28 -> 14x14
                "conv2": nn.Conv2d(64, 64, kernel_size=3, padding=1),
                "norm2": nn.GroupNorm(8, 64),
                "relu2": nn.ReLU(),
                "pool2": nn.MaxPool2d(kernel_size=2),  # 14x14 -> 7x7
                "conv3": nn.Conv2d(64, 64, kernel_size=3, padding=1),
                "norm3": nn.GroupNorm(8, 64),
                "relu3": nn.ReLU(),
                "pool3": nn.MaxPool2d(kernel_size=2),  # 7x7 -> 3x3
                "conv4": nn.Conv2d(64, 64, kernel_size=3, padding=1),
                "norm4": nn.GroupNorm(8, 64),
                "relu4": nn.ReLU(),
                "pool4": nn.MaxPool2d(kernel_size=2),  # 3x3 -> 1x1
                "average": nn.AdaptiveAvgPool2d(1),
                "flatten": nn.Flatten(),
                "fc": nn.Linear(self.final_feature_dim, n_way),
            }
        )

        # Register the layers so named_parameters() includes their weights.
        for name, layer in self.layers.items():
            if isinstance(layer, nn.Module):
                self.add_module(name, layer)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        """Run the network with its registered parameters."""
        for name, layer in self.layers.items():
            x = layer(x)
        return x

    def functional_forward(self, x, fast_params):
        """Run the network with the supplied parameter mapping."""
        x = F.conv2d(x, fast_params["conv1.weight"], fast_params["conv1.bias"], padding=1)
        x = F.group_norm(x, 8, fast_params["norm1.weight"], fast_params["norm1.bias"])
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = F.conv2d(x, fast_params["conv2.weight"], fast_params["conv2.bias"], padding=1)
        x = F.group_norm(x, 8, fast_params["norm2.weight"], fast_params["norm2.bias"])
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = F.conv2d(x, fast_params["conv3.weight"], fast_params["conv3.bias"], padding=1)
        x = F.group_norm(x, 8, fast_params["norm3.weight"], fast_params["norm3.bias"])
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = F.conv2d(x, fast_params["conv4.weight"], fast_params["conv4.bias"], padding=1)
        x = F.group_norm(x, 8, fast_params["norm4.weight"], fast_params["norm4.bias"])
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = torch.flatten(F.adaptive_avg_pool2d(x, 1), 1)

        x = F.linear(x, fast_params["fc.weight"], fast_params["fc.bias"])

        return x


## 内循环与外循环

**内循环**从共享初始参数出发，在一个任务的支持集上做几步梯度下降。**外循环**用适应后的参数计算查询集损失，对一批任务取平均，再更新共享初始参数。内循环学习当前任务，外循环改善下一次适应的起点。

代码中的 `fast_params` 最初引用模型参数，每次内循环更新都会生成新的参数张量。MAML 保留支持集梯度的计算图，使外循环能继续对适应过程求导。FO-MAML 设置 `create_graph=False`，忽略支持集梯度随参数变化的部分，但仍保留初始参数到更新后参数的直接梯度路径。

`n_meta_updates` 指定外循环更新次数，每次更新都重新采样一批任务。训练曲线记录每批任务适应后的查询集表现；每隔 200 次更新，在固定验证任务上比较适应 0、1、5 步的准确率。训练结束后，加载五步验证准确率最高时的参数，再进行测试。


In [ ]:
def train_maml(
    is_fomaml,
    train_ds,
    val_ds,
    initial_state,
    n_way,
    k_shot,
    q_query,
    n_meta_updates=2000,
    n_tasks_per_batch=8,
    inner_lr=0.1,
    meta_lr=0.001,
    n_inner_steps=5,
    validation_steps=5,
    validation_every=200,
    validation_episodes=100,
    print_every=100,
):
    """Train with matched initialization and task sampling; select on validation classes."""
    seed_everything(SEED, deterministic=True)
    model = MAML_CNN(
        n_way=n_way, input_channels=train_ds.input_channels, img_size=train_ds.img_size
    ).to(device)
    model.load_state_dict(initial_state)
    model.initial_digest = parameter_digest(model)
    train_rng = random.Random(SEED + 1)
    optimizer = optim.Adam(model.parameters(), lr=meta_lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_meta_updates, eta_min=1e-4)
    history, validation_history = [], []
    best_accuracy, best_state, best_update = -1.0, None, None
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    print(f"{'FO-MAML' if is_fomaml else 'MAML'}: initialization {model.initial_digest[:12]}")
    for update in range(n_meta_updates):
        model.train()
        optimizer.zero_grad()
        loss_sum, accuracy_sum = 0.0, 0.0
        for _ in range(n_tasks_per_batch):
            support_images, support_labels, query_images, query_labels = train_ds.sample_episode(
                n_way, k_shot, q_query, rng=train_rng
            )
            support_images, support_labels, query_images, query_labels = [
                value.to(device)
                for value in (support_images, support_labels, query_images, query_labels)
            ]
            fast_params = OrderedDict(model.named_parameters())
            for _ in range(n_inner_steps):
                loss = F.cross_entropy(
                    model.functional_forward(support_images, fast_params), support_labels
                )
                grads = torch.autograd.grad(loss, fast_params.values(), create_graph=not is_fomaml)
                fast_params = OrderedDict(
                    (name, value - inner_lr * grad)
                    for (name, value), grad in zip(fast_params.items(), grads)
                )
            logits = model.functional_forward(query_images, fast_params)
            query_loss = F.cross_entropy(logits, query_labels)
            # Task-wise backward is equivalent to the mean loss, without retaining every graph.
            (query_loss / n_tasks_per_batch).backward()
            loss_sum += query_loss.item()
            accuracy_sum += (logits.argmax(1) == query_labels).float().mean().item()
        optimizer.step()
        scheduler.step()
        history.append(
            dict(
                update=update + 1,
                loss=loss_sum / n_tasks_per_batch,
                accuracy=accuracy_sum / n_tasks_per_batch,
            )
        )
        if (update + 1) % print_every == 0:
            print(
                f"Update {update + 1}: query loss={history[-1]['loss']:.4f}, "
                f"accuracy={history[-1]['accuracy']:.2%}"
            )
        if (update + 1) % validation_every == 0 or update + 1 == n_meta_updates:
            scores = evaluate_adaptation(
                model,
                val_ds,
                n_way,
                k_shot,
                q_query,
                inner_lr,
                steps=tuple(sorted({0, 1, 5, validation_steps})),
                n_episodes=validation_episodes,
                seed=SEED + 2,
            )
            validation_history.append(dict(update=update + 1, scores=scores))
            accuracy = scores[validation_steps]["accuracy"]
            print(f"Validation update {update + 1}: {scores}")
            if accuracy > best_accuracy:
                best_accuracy, best_update = accuracy, update + 1
                best_state = {
                    name: value.detach().clone() for name, value in model.state_dict().items()
                }
    model.load_state_dict(best_state)
    model.training_history = history
    model.validation_history = validation_history
    model.selected_update = best_update
    if device.type == "cuda":
        torch.cuda.synchronize()
    model.elapsed_seconds = time.perf_counter() - start_time
    print(f"Selected update {best_update} by validation; elapsed {model.elapsed_seconds:.1f}s")
    plot_meta_history(history, validation_history, n_inner_steps)
    return model


### 观察外循环的学习过程

训练结束后，用下面的函数绘制查询集损失和准确率。验证曲线分别显示适应 0、1、5 步的结果，用来观察共享初始参数及适应后表现随更新次数的变化。


In [ ]:
def plot_meta_history(history, validation_history, n_inner_steps):
    """Plot training queries and the fixed validation adaptation curves."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot([row["update"] for row in history], [row["loss"] for row in history])
    axes[0].set_ylabel("Training query cross entropy")
    axes[1].plot(
        [row["update"] for row in history],
        [row["accuracy"] for row in history],
        alpha=0.4,
        label="training, 1-step" if n_inner_steps == 1 else "training",
    )
    for step in sorted(validation_history[-1]["scores"]):
        axes[1].plot(
            [row["update"] for row in validation_history],
            [row["scores"][step]["accuracy"] for row in validation_history],
            marker="o",
            label=f"validation, {step}-step",
        )
    axes[1].set_ylabel("Query accuracy")
    axes[1].legend()
    for ax in axes:
        ax.set_xlabel("Meta-update")
    plt.tight_layout()
    plt.show()


## 在新类别上评估

每个评估任务都从元训练得到的初始参数重新开始，只用该任务的支持集适应，再计算查询集准确率。一个任务的适应结果不会传给下一个任务。

下面比较适应前、1 步和 5 步后的平均准确率，并给出近似 95% 置信区间。区间根据各测试任务的准确率计算，反映任务采样带来的波动；训练种子改变后的波动需要通过多次重新训练来观察。

评估时，支持集上的适应仍需要求梯度，但不再进行外循环更新，因此无需保留用于二阶求导的计算图。


In [ ]:
def evaluate_adaptation(
    model, dataset, n_way, k_shot, q_query, inner_lr, steps=(0, 1, 5), n_episodes=100, seed=1
):
    """Score the same query images at each adaptation step without updating the model."""
    was_training = model.training
    model.eval()
    rng = random.Random(seed)
    scores = {step: [] for step in steps}
    model_device = next(model.parameters()).device
    for _ in range(n_episodes):
        support_images, support_labels, query_images, query_labels = dataset.sample_episode(
            n_way, k_shot, q_query, rng=rng
        )
        support_images, support_labels, query_images, query_labels = [
            value.to(model_device)
            for value in (support_images, support_labels, query_images, query_labels)
        ]
        fast_params = OrderedDict(model.named_parameters())
        for step in range(max(steps) + 1):
            if step in scores:
                with torch.no_grad():
                    logits = model.functional_forward(query_images, fast_params)
                    scores[step].append((logits.argmax(1) == query_labels).float().mean().item())
            if step < max(steps):
                loss = F.cross_entropy(
                    model.functional_forward(support_images, fast_params), support_labels
                )
                grads = torch.autograd.grad(loss, fast_params.values())
                fast_params = OrderedDict(
                    (name, value - inner_lr * grad)
                    for (name, value), grad in zip(fast_params.items(), grads)
                )
    model.train(was_training)
    return {
        step: dict(
            accuracy=float(np.mean(values)),
            ci95=float(1.96 * np.std(values, ddof=1) / np.sqrt(n_episodes))
            if n_episodes > 1
            else 0.0,
        )
        for step, values in scores.items()
    }


### 汇总测试任务的准确率

训练完成后，在固定的一组测试任务上比较适应前后的准确率，并报告任务间波动得到的置信区间。下面的 `test_model()` 调用同一个适应过程，方便两种方法使用相同的测试设置。


In [ ]:
def test_model(
    model,
    title_prefix,
    val_ds,
    n_way,
    k_shot,
    q_query,
    n_inner_steps=5,
    inner_lr=0.1,
    n_test_episodes=600,
):
    """Report a prespecified adaptation curve on a common set of final test episodes."""
    scores = evaluate_adaptation(
        model,
        val_ds,
        n_way,
        k_shot,
        q_query,
        inner_lr,
        steps=tuple(sorted({0, 1, n_inner_steps})),
        n_episodes=n_test_episodes,
        seed=SEED + 3,
    )
    for step, result in scores.items():
        print(
            f"{title_prefix}, steps={step}: accuracy={result['accuracy']:.2%} "
            f"± {result['ci95']:.2%} (95% episode CI; {n_test_episodes} episodes)"
        )
    return scores


## 设置任务大小与训练次数

默认使用 5-way 1-shot 任务：支持集共 5 张图像，查询集每类 15 张，共 75 张。每次外循环采样 8 个任务，各在支持集上更新 5 步。

MAML 和 FO-MAML 各训练 2000 次外循环。训练中使用 100 个固定验证任务选择参数，最终在同一组 600 个测试任务上评估。

模型直接优化五步适应后的查询集损失，所以重点比较两种方法适应五步的效果。适应前和一步后的结果可以帮助观察提升发生在哪个阶段。支持集很小时，继续更新也可能过拟合，查询集准确率未必随步数增加而提高。


In [ ]:
N_WAY = 5  # Classes per episode.
K_SHOT = 1  # Support examples per class.
Q_QUERY = 15  # Query examples per class.
INNER_LR = 0.1
META_LR = 0.001

# Support-set updates per training episode.
N_INNER_STEPS_TRAIN = 5
# Support-set updates per evaluation episode.
N_INNER_STEPS_TEST = 5
N_TASKS_PER_BATCH = 8

N_TEST_EPISODES = 600
N_VALIDATION_EPISODES = 100
seed_everything(SEED, deterministic=True)
initial_model = MAML_CNN(
    n_way=N_WAY, input_channels=train_ds.input_channels, img_size=train_ds.img_size
)
initial_state = {name: value.detach().clone() for name, value in initial_model.state_dict().items()}
initial_digest = parameter_digest(initial_model)


## 运行 MAML

训练曲线展示已知训练类别上的学习过程，验证和测试使用未见过的类别。可以对照观察：训练任务的查询集损失下降时，新类别上的适应效果是否也在改善？


In [ ]:
N_META_UPDATES_MAML = 2000

maml_model = train_maml(
    is_fomaml=False,
    train_ds=train_ds,
    val_ds=val_ds,
    initial_state=initial_state,
    n_way=N_WAY,
    k_shot=K_SHOT,
    q_query=Q_QUERY,
    n_meta_updates=N_META_UPDATES_MAML,
    n_tasks_per_batch=N_TASKS_PER_BATCH,
    inner_lr=INNER_LR,
    meta_lr=META_LR,
    n_inner_steps=N_INNER_STEPS_TRAIN,
    validation_steps=N_INNER_STEPS_TEST,
    validation_episodes=N_VALIDATION_EPISODES,
    print_every=100,
)

assert maml_model.initial_digest == initial_digest


## 运行 FO-MAML

下面保持网络、任务大小和更新次数相同，将 MAML 换为一阶近似。两种方法复制相同的初始权重，并使用相同顺序的训练任务；验证使用独立采样器，不打断训练采样。

两种方法都依据五步验证准确率选择模型，最后在相同测试任务上比较准确率和耗时。这组对照使用一个训练种子，反映本次运行中的差异。


In [ ]:
N_META_UPDATES_FOMAML = 2000

fomaml_model = train_maml(
    is_fomaml=True,
    train_ds=train_ds,
    val_ds=val_ds,
    initial_state=initial_state,
    n_way=N_WAY,
    k_shot=K_SHOT,
    q_query=Q_QUERY,
    n_meta_updates=N_META_UPDATES_FOMAML,
    n_tasks_per_batch=N_TASKS_PER_BATCH,
    inner_lr=INNER_LR,
    meta_lr=META_LR,
    n_inner_steps=N_INNER_STEPS_TRAIN,
    validation_steps=N_INNER_STEPS_TEST,
    validation_episodes=N_VALIDATION_EPISODES,
    print_every=100,
)

assert fomaml_model.initial_digest == initial_digest
# Evaluate the fixed selected models on identical held-out episodes.
maml_test_scores = test_model(
    maml_model,
    "MAML",
    test_ds,
    N_WAY,
    K_SHOT,
    Q_QUERY,
    n_inner_steps=N_INNER_STEPS_TEST,
    inner_lr=INNER_LR,
    n_test_episodes=N_TEST_EPISODES,
)
fomaml_test_scores = test_model(
    fomaml_model,
    "FO-MAML",
    test_ds,
    N_WAY,
    K_SHOT,
    Q_QUERY,
    n_inner_steps=N_INNER_STEPS_TEST,
    inner_lr=INNER_LR,
    n_test_episodes=N_TEST_EPISODES,
)


## 比较不同的 N-way K-shot 设置

下面在 Omniglot 上比较 5/20-way × 1/5-shot，在 Mini-ImageNet 上比较 5/10-way × 1/5-shot。N 越大，需要区分的类别越多；K 越大，每类可用于适应的带标签样本越多。

每种设置都重新训练 MAML 与 FO-MAML，两种方法共享初始参数及训练、验证和测试任务。不同 N/K 设置分别采样任务，因此逐任务的配对比较只适用于同一设置下的两种方法。

将 `RUN_TASK_MATRIX` 设为 `True` 即可运行全部组合。每种方法默认训练 2000 次外循环，每次 8 个任务，各适应 5 步，耗时会明显长于上面的单组示例。若只想先检查流程，可以临时将 `MATRIX_UPDATES` 改为 2；这不足以判断学习效果。Mini-ImageNet 的 2000 次更新是本课程的起始设置，与论文实验配置不同。

输出列出适应前后的准确率、随机猜测水平，以及包含验证的训练时间。增大 N 或 K 会增加每次更新处理的图像数，因此相同更新次数下的耗时也会变化。


In [ ]:
def compare_task_sizes(
    configurations,
    updates=2000,
    tasks_per_batch=8,
    validation_episodes=100,
    test_episodes=600,
    q_query=15,
):
    rows = []
    for dataset_name in dict.fromkeys(name for name, _, _ in configurations):
        train_data, validation_data, test_data = load_few_shot_data(dataset_name)
        for name, n_way, k_shot in configurations:
            if name != dataset_name:
                continue
            seed_everything(SEED, deterministic=True)
            initial = MAML_CNN(n_way, train_data.input_channels, train_data.img_size).state_dict()
            for first_order in (False, True):
                model = train_maml(
                    first_order,
                    train_data,
                    validation_data,
                    initial,
                    n_way,
                    k_shot,
                    q_query,
                    n_meta_updates=updates,
                    n_tasks_per_batch=tasks_per_batch,
                    n_inner_steps=5,
                    validation_steps=5,
                    validation_episodes=validation_episodes,
                )
                scores = test_model(
                    model,
                    f"{name} {n_way}-way {k_shot}-shot",
                    test_data,
                    n_way,
                    k_shot,
                    q_query,
                    n_test_episodes=test_episodes,
                )
                rows.append(
                    dict(
                        dataset=name,
                        n_way=n_way,
                        k_shot=k_shot,
                        method="FO-MAML" if first_order else "MAML",
                        initial_digest=model.initial_digest,
                        selected_update=model.selected_update,
                        seconds=model.elapsed_seconds,
                        scores=scores,
                    )
                )
                print(
                    f"{rows[-1]['method']} {name} {n_way}-way {k_shot}-shot: "
                    f"chance={1 / n_way:.2%}, before={scores[0]['accuracy']:.2%}, "
                    f"after={scores[5]['accuracy']:.2%} ± {scores[5]['ci95']:.2%}, "
                    f"time={model.elapsed_seconds:.1f}s"
                )
    return rows


TASK_CONFIGURATIONS = (
    ("omniglot", 5, 1),
    ("omniglot", 5, 5),
    ("omniglot", 20, 1),
    ("omniglot", 20, 5),
    ("mini-imagenet", 5, 1),
    ("mini-imagenet", 5, 5),
    ("mini-imagenet", 10, 1),
    ("mini-imagenet", 10, 5),
)
MATRIX_UPDATES = 2000
RUN_TASK_MATRIX = False
if RUN_TASK_MATRIX:
    task_size_results = compare_task_sizes(TASK_CONFIGURATIONS, updates=MATRIX_UPDATES)


## 可选扩展

- 可以改变 N 或 K，观察候选类别增多、支持样本增多时，适应前后的准确率如何变化。同一设置下保持 MAML 与 FO-MAML 的训练和评估任务一致。
- 可以尝试 Reptile：每个任务在支持集上更新多步，再让共享初始化向任务参数移动。它的外循环不使用 MAML 的查询损失；比较时沿用相同数据划分，并留意实际处理的图像数和更新时间。
